In [1]:
"""
train_model.py — LSTM Model Training for EV Registration Forecasting
=====================================================================
Trains a 2-layer LSTM on monthly WA EV registration data.
Outputs:
  - ev_model.h5   (trained Keras model)
  - scaler.pkl    (fitted MinMaxScaler)
  - metrics.json  (MAPE, RMSE for the dashboard)

Usage:
  python train_model.py
"""

import json
import os
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")



In [2]:
# ──────────────────────────────────────────────────────────────
# 1. Load & Prepare Data


In [3]:
# ──────────────────────────────────────────────────────────────
DATA_PATH = "ev_population.csv"

LOOKBACK = 12  # months

print("=" * 60)
print("  EV Registration Forecasting — Model Training Pipeline")
print("=" * 60)

df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)
values = df["Electric Vehicle (EV) Total"].values.astype(float).reshape(-1, 1)

print(f"\n[DATA] Dataset: {len(values)} monthly records")
print(f"   Range : {df['Date'].min().strftime('%b %Y')} -> {df['Date'].max().strftime('%b %Y')}")
print(f"   Min   : {int(values.min()):,}")
print(f"   Max   : {int(values.max()):,}")



  EV Registration Forecasting — Model Training Pipeline

[DATA] Dataset: 113 monthly records
   Range : Jan 2017 -> May 2026
   Min   : 8,348
   Max   : 313,512


In [4]:
# ──────────────────────────────────────────────────────────────
# 2. Scale


In [5]:
# ──────────────────────────────────────────────────────────────
scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(values)



In [6]:
# ──────────────────────────────────────────────────────────────
# 3. Create Sequences


In [7]:
# ──────────────────────────────────────────────────────────────
def create_sequences(data, lookback):
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i - lookback : i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled, LOOKBACK)
X = X.reshape(X.shape[0], X.shape[1], 1)  # (samples, timesteps, features)

# Train/Test split — last 20% for testing
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"\n[SPLIT] Train/Test Split:")
print(f"   Training : {len(X_train)} samples")
print(f"   Testing  : {len(X_test)} samples")
print(f"   Lookback : {LOOKBACK} months")




[SPLIT] Train/Test Split:
   Training : 80 samples
   Testing  : 21 samples
   Lookback : 12 months


In [8]:
# ──────────────────────────────────────────────────────────────
# 4. Build LSTM Model


In [9]:
# ──────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

tf.random.set_seed(42)
np.random.seed(42)

model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(LOOKBACK, 1)),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(1),
])

model.compile(optimizer="adam", loss="mse")

print("\n[MODEL] Architecture:")
model.summary()




[MODEL] Architecture:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 12, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 12, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,857 (116.63 KB)

 Trainable params: 29,857 (116.63 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
# ──────────────────────────────────────────────────────────────
# 5. Train


In [11]:
# ──────────────────────────────────────────────────────────────
print("\n[TRAIN] Training started...")
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_data=(X_test, y_test),
    verbose=1,
)




[TRAIN] Training started...
Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 103ms/step - loss: 0.0584 - val_loss: 0.4843
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0310 - val_loss: 0.2460
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0145 - val_loss: 0.0482
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0115 - val_loss: 0.0270
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0066 - val_loss: 0.0566
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0057 - val_loss: 0.0395
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0024 - val_loss: 5.5004e-04
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.0010 - val_loss: 0.0147
Epoch 9/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0016 - val_loss: 0.0076
Epoch 10/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.0010 - val_loss: 0.0021
Epoch 11/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0014 - val_loss: 0.0043
Epoch 12/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s

In [12]:
# ──────────────────────────────────────────────────────────────
# 6. Evaluate


In [13]:
# ──────────────────────────────────────────────────────────────
predictions_scaled = model.predict(X_test)
predictions = scaler.inverse_transform(predictions_scaled)
actuals = scaler.inverse_transform(y_test.reshape(-1, 1))

# MAPE
mape = np.mean(np.abs((actuals - predictions) / actuals)) * 100
# RMSE
rmse = np.sqrt(np.mean((actuals - predictions) ** 2))

print("\n" + "=" * 60)
print("  [RESULTS] Evaluation Results")
print("=" * 60)
print(f"   MAPE : {mape:.2f}%")
print(f"   RMSE : {rmse:,.0f}")
print(f"   {'[OK] MAPE < 10% -- Target Met!' if mape < 10 else '[WARN] MAPE >= 10% -- Consider tuning.'}")



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 229ms/step

  [RESULTS] Evaluation Results
   MAPE : 9.54%
   RMSE : 27,067
   [OK] MAPE < 10% -- Target Met!


In [14]:
# ──────────────────────────────────────────────────────────────
# 7. Save Artifacts


In [15]:
# ──────────────────────────────────────────────────────────────
base_dir = "."

model_path = os.path.join(base_dir, "ev_model.h5")
scaler_path = os.path.join(base_dir, "scaler.pkl")
metrics_path = os.path.join(base_dir, "metrics.json")

model.save(model_path)
joblib.dump(scaler, scaler_path)

metrics = {"mape": round(mape, 2), "mae": round(mae, 2), "rmse": round(rmse, 2)}
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\n[SAVE] Saved Artifacts:")
print(f"   Model   -> {model_path}")
print(f"   Scaler  -> {scaler_path}")
print(f"   Metrics -> {metrics_path}")
print("\n[DONE] Training complete. Run `streamlit run app.py` to launch the dashboard.")



[SAVE] Saved Artifacts:
   Model   -> .\ev_model.h5
   Scaler  -> .\scaler.pkl
   Metrics -> .\metrics.json

[DONE] Training complete. Run `streamlit run app.py` to launch the dashboard.
